#### Coding an LLM architecture
Large language models (LLMs) such as GPT (Generative Pretrained Transformer) are deep neural network architectures designed to generate text one token at a time. Despite their large scale, the overall architecture is relatively simple, as many of its core components are repeated multiple times.

For clarity and educational purposes, earlier examples used small embedding sizes. We now scale up the implementation to match the size of a small GPT-2 model with approximately 124 million parameters. Larger GPT-2 variants will be addressed later by loading pretrained weights and adapting the same architecture.

In the context of deep learning, model parameters refer to the trainable weights that are optimized during training to minimize a loss function. These parameters define the model’s capacity to learn patterns from data

<div style="text-align: center; margin-top: 20px;">
  <img 
    src="https://raw.githubusercontent.com/salavii/llm-from-scratch/main/images/llm.png"
    style="width: 800px; border-radius: 10px; display: block; margin-left: auto; margin-right: auto;"
  >


### GPT-2 Small Configuration
The following dictionary defines the configuration of the small GPT-2 model (approximately 124 million parameters). These values are taken directly from the original GPT-2 architecture and will be used throughout the implementation

In [71]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Size of the tokenizer vocabulary 
    "context_length": 1024,  # Maximum number of input tokens
    "emb_dim": 768,          # Token embedding dimension
    "n_heads": 12,           # Number of attention heads per layer
    "n_layers": 12,          # Number of transformer blocks
    "drop_rate": 0.1,        # Dropout rate for regularization
    "qkv_bias": False        # Whether to use bias in QKV projections
}


### Building a Placeholder GPT Architecture

Using the defined configuration, we begin by implementing a placeholder GPT architecture called DummyGPTModel. This simplified model provides a high-level view of how the different components of a GPT model fit together and helps identify which additional components need to be implemented to assemble the full architecture

We start with step 1: implementing a basic GPT backbone (DummyGPTModel) that serves as a foundation for progressively adding more detailed components in later stages

In [72]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # Create a stack of n_layers Transformer blocks that process the sequence step by step
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        
        # Final normalization before the output layer
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
                                    cfg["emb_dim"], cfg["vocab_size"], bias=False
                                 )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
                                torch.arange(seq_len, device=in_idx.device)
                                 )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

class DummyTransformerBlock(nn.Module):   
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):    
         return x

class DummyLayerNorm(nn.Module):          
    def __init__(self, normalized_shape, eps=1e-5):   
        super().__init__()
        
    def forward(self, x):
        return x

### Overview of the Dummy GPT Model Architecture
The `DummyGPTModel` class defines a simplified GPT-style architecture using PyTorch’s `nn.Module`. The model consists of token and positional embeddings, dropout, a stack of transformer blocks `(DummyTransformerBlock)`, a final layer normalization `(DummyLayerNorm)`, and a linear output head.

The model configuration is provided via a Python dictionary, such as the previously defined `GPT_CONFIG_124M`. The `forward` method specifies how data flows through the model: token and positional embeddings are computed, dropout is applied, the data is processed through the transformer blocks, normalized, and finally projected to vocabulary-sized logits by the output layer.

Although the code is fully functional, placeholder implementations are used for the transformer blocks and layer normalization. These components will be replaced with their full implementations later. This placeholder architecture allows us to understand the overall structure of a GPT model before focusing on individual components in detail

In the next step, we prepare input data and initialize a GPT model to demonstrate how tokenized text flows into and out of the model

In [73]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


Next, we initialize a new 124-million-parameter DummyGPTModel instance and feed it
the tokenized batch: 

In [74]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


## Normalizing Activations with Layer Normalization
Training deep neural networks with many layers can be challenging due to issues such as vanishing or exploding gradients. These problems lead to unstable training dynamics and make it difficult for the network to adjust its weights effectively, which hinders the learning process.

Layer normalization addresses this problem by normalizing the activations of a layer so that they have a mean of 0 and a variance of 1. This normalization stabilizes the flow of values through the network, accelerates convergence, and results in more reliable training.

In GPT-2 and modern transformer architectures, layer normalization is typically applied before and after the multi-head attention module, as well as before the final output layer. In this chapter, we replace the placeholder `DummyLayerNorm` with a real layer normalization implementation to improve training stability and efficiency

<br/>

#### Example: Neural Network Layer with 5 Inputs and 6 Outputs#### 
To illustrate how a neural network layer operates, we recreate the example shown in Figure 4.5 by implementing a layer with five input features and six output features. This layer is then applied to two input examples

In [75]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


In [76]:
out.shape

torch.Size([2, 6])

Before we apply layer normalization to these outputs, let’s examine the mean and
variance

In [77]:
mean = out.mean(dim=-1, keepdim= True)
var = out.var(dim=-1, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


 Next, let’s apply layer normalization to the layer outputs we obtained earlier. The
operation consists of subtracting the mean and dividing by the square root of the vari
ance (also known as the standard deviation)

In [78]:
out_norm = (out - mean) / torch.sqrt(var)
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Normalized layer outputs:\n", out_norm)
print("Mean:\n", mean)
print("Variance:\n", var)

Normalized layer outputs:
 tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
Mean:
 tensor([[0.0000],
        [0.0000]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


## Numerical Precision and Scientific Notation

Values such as -5.9605e-08 are expressed in scientific notation.

This value is extremely close to zero but not exactly zero due to finite numerical precision.

Small floating-point errors are normal in deep learning computations.

To improve readability when printing tensors, scientific notation can be disabled:

> torch.set_printoptions(sci_mode=False)


This does not change the computation—only how numbers are displayed

In [79]:
torch.set_printoptions(sci_mode=False)
print("Mean= \n", mean)
print("Var= \n", var)

Mean= 
 tensor([[0.0000],
        [0.0000]], grad_fn=<MeanBackward1>)
Var= 
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


<br/>

So far, we implemented layer normalization manually (compute mean/variance, normalize outputs).
Next, we wrap this logic inside a custom nn.Module so it can be reused like any other PyTorch layer

In [80]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var - self.eps)
        return self.scale * norm_x + self.shift

In [81]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)

mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)


Mean:
 tensor([[-0.0000],
        [ 0.0000]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


## Layer Normalization vs. Batch Normalization

> **Batch Normalization** normalizes activations across the batch dimension.

> **Layer Normalization** normalizes activations across the feature dimension.

Since large language models often operate with variable or small batch sizes due to hardware constraints, batch normalization becomes less effective and unstable.

Layer normalization normalizes each input independently of the batch size, making it more flexible and stable for:

- Large-scale models
- Distributed training
- Resource-constrained deployment

As a result, LayerNorm is the standard normalization method in Transformer-based architectures

# Implementing a feed forward network with GELU activations
In Transformer-based LLMs, each Transformer block contains a small **feed-forward network (FFN/MLP) submodule**.
We start by implementing the GELU activation function, which is widely used in GPT-style models.

Why GELU instead of ReLU?
ReLU is simple and effective, but modern LLMs often use smoother activations such as:
- GELU (Gaussian Error Linear Unit)
- SwiGLU (Swish-Gated Linear Unit)

These functions are smoother and can improve training and performance in deep models compared to ReLU

In [82]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
        (x + 0.044715 * torch.pow(x, 3))
                                        ))

In [83]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)
    

In [84]:
ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2, 3, 768)         
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


### Role of the FeedForward Network in GPT

The FeedForward module is a critical component that enhances the model’s capacity to learn and generalize.

Although its input and output dimensions are identical, the module:

1. Expands the embedding dimension into a higher-dimensional space using the first linear layer

2. Applies a nonlinear GELU activation

3. Projects the representation back to the original embedding dimension with a second linear layer

This expand–nonlinear–contract design enables the model to explore a richer and more expressive representation space while remaining compatible with residual connections

# Adding shortcut connections
Shortcut connections, also known as skip or residual connections, were originally introduced to address the vanishing gradient problem in deep neural networks.

A shortcut connection adds the output of an earlier layer directly to the output of a later layer, creating a shorter path for gradients during backpropagation.

This design:

- Preserves gradient flow
- Improves training stability
- Enables very deep architectures, such as GPT-style Transformer models

Residual connections are a key component of modern deep learning architectures

In [85]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()

        self.use_shortcut = use_shortcut

        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU()),
        ])

    def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer
            layer_output = layer(x)

            # Apply shortcut connection if enabled and shapes match
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output

        return x


### Initializing a Deep Network Without Shortcut Connections

This code defines a deep neural network with five layers, where each layer is:

`Linear(...)` followed by `GELU()`

During the forward pass, the input is passed through all layers sequentially.
If `self.use_shortcut=True` and shapes match, a residual connection is applied `(x = x + layer_output)`. Otherwise, the layer output replaces x.

In this step, we initialize the network without shortcut connections:

Intermediate layers map 3 → 3

The final layer maps 3 → 1

So the overall architecture is:
`3 → 3 → 3 → 3 → 3 → 1`

In [86]:
layer_sizes = [3, 3, 3, 3, 3, 1]  
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123)                           
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)


Next, we implement a function that computes the gradients in the model’s back
ward pass:

In [87]:
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])

    loss = nn.MSELoss()
    loss = loss(output, target)

    loss.backward()

    for name, param in model.named_parameters():
        if "weight" in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [88]:
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173590746708214
layers.1.0.weight has gradient mean of 0.0001201116101583466
layers.2.0.weight has gradient mean of 0.0007152042235247791
layers.3.0.weight has gradient mean of 0.0013988739810883999
layers.4.0.weight has gradient mean of 0.00504964729771018


When running print_gradients on the model without shortcut connections, gradient magnitudes decrease from the last layer to the first, indicating the `vanishing gradient` problem.

Shortcut (residual) connections add a direct path for gradient flow, allowing gradients to reach earlier layers more effectively. This improves training stability and enables deep architectures like GPT to learn successfully

In [89]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22169791162014008
layers.1.0.weight has gradient mean of 0.20694102346897125
layers.2.0.weight has gradient mean of 0.32896995544433594
layers.3.0.weight has gradient mean of 0.2665732204914093
layers.4.0.weight has gradient mean of 1.3258541822433472


## Shortcut Connections and Gradient Stability
With shortcut connections enabled, gradient values remain stable as they propagate toward earlier layers, instead of vanishing. Although the final layer still exhibits the largest gradient, residual connections prevent gradients from shrinking to near-zero values in early layers.

Shortcut connections are therefore essential for overcoming the vanishing gradient problem and are a core component of large-scale models such as LLMs. They enable stable and effective training by ensuring consistent gradient flow across layers.

Next, all previously introduced components—layer normalization, GELU activation, feed-forward networks, and shortcut connections—are combined into a Transformer block, the final building block of the GPT architecture

## We can create the TransformerBlock in code.

In [91]:
from Attention import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.att = MultiHeadAttention(
             d_in=cfg["emb_dim"],
             d_out=cfg["emb_dim"],
             context_length=cfg["context_length"],
             num_heads=cfg["n_heads"], 
             dropout=cfg["drop_rate"],
             qkv_bias=cfg["qkv_bias"]
                                    )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x
        
        

## TransformerBlock: Pre-LayerNorm + Residual Connections
A `TransformerBlock` in GPT-style models typically contains two submodules:

1. Masked Multi-Head Self-Attention
2. Feed-Forward Network (MLP)

In this implementation, **LayerNorm is applied before** each submodule (Pre-LayerNorm), and **dropout is applied after** each submodule for regularization. Pre-LN is commonly preferred in modern LLMs because it tends to produce more stable training dynamics than the older Post-LayerNorm design.

The forward pass also includes **residual (shortcut) connections**, where the input is added to the output of each submodule. Residual connections help preserve gradient flow and are crucial for training deep Transformer-based models like GPT

In [97]:
torch.manual_seed(123)
x = torch.rand(2, 4, 768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print(f"x shape = {x.shape} \noutput shape = {output.shape}")

x shape = torch.Size([2, 4, 768]) 
output shape = torch.Size([2, 4, 768])


<div style="text-align: center; margin-top: 20px;">
  <img 
    src="https://raw.githubusercontent.com/salavii/llm-from-scratch/main/images/Screenshot 2026-01-05 142031.png"
    style="width: 750px; border-radius: 10px; display: block; margin-left: auto; margin-right: auto;"
  >


# Coding the GPT model
We initially implemented a high-level placeholder GPT architecture (DummyGPTModel) to visualize input/output flow while keeping key components as black boxes (DummyTransformerBlock, DummyLayerNorm).

Now we replace these placeholders with the real implementations:

TransformerBlock (masked multi-head attention + feed-forward + dropout + residuals, using Pre-LayerNorm)

LayerNorm

In GPT-2 124M, the Transformer block is stacked 12 times, controlled by GPT_CONFIG_124M["n_layers"]. Larger GPT-2 variants use more blocks (e.g., 48 layers for the 1.5B model).

After the final Transformer block, the model applies a final layer normalization and then a linear output head that projects the hidden states to vocabulary-sized logits (e.g., 50,257 for GPT-2) for next-token prediction

<div style="text-align: center; margin-top: 20px;">
  <img 
    src="https://raw.githubusercontent.com/salavii/llm-from-scratch/main/images/GPT1.png"
    style="width: 750px; border-radius: 10px; display: block; margin-left: auto; margin-right: auto;"
  >


In [98]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
                                       )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
                                 )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits

Thanks to the `TransformerBlock` abstraction, the `GPTModel` class remains compact and mostly focuses on wiring components together.

**Initialization `(__init__)`:**

- Creates token and positional embedding layers from a configuration dictionary cfg.
- Builds a stack of TransformerBlock modules of length cfg["n_layers"].
- Applies a final LayerNorm after the transformer stack to stabilize training.
- Defines a bias-free linear output head that projects hidden states from emb_dim to vocab_size, producing logits.

**Forward pass (`forward`)**:
- Takes a batch of token indices (batch_size, seq_len).
- Computes token embeddings and adds positional embeddings.
- Passes the sequence through the transformer blocks.
- Applies final normalization.
- Projects to vocabulary logits, which represent unnormalized scores for next-token prediction.

Next, we instantiate the GPT-2 124M-style model using GPT_CONFIG_124M and feed it a batch of tokenized input

In [99]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2711,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


The model output has shape [2, 4, 50257] because we passed a batch of 2 input sequences with 4 tokens each. The last dimension (50257) corresponds to the tokenizer’s vocabulary size, meaning each token position outputs a vector of vocabulary-sized logits.

Before converting logits back into tokens, we can analyze the model size by counting parameters. The numel() method (“number of elements”) returns how many scalar values are stored in a tensor. Summing numel() across all parameter tensors gives the total number of model parameters

In [101]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


### Why 124M vs 163M Parameters? (Short Summary)

The mismatch is due to weight tying.

In the original GPT-2 architecture:

The token embedding layer and the output (linear) layer share the same weights.

In this implementation:

These two layers are separate

Their parameters are therefore counted twice

🔹 Shapes

Token embedding: (vocab_size, emb_dim)

Output layer: (emb_dim, vocab_size)

🔹 Result

Without weight tying → ~163M parameters

With weight tying (GPT-2) → ~124M parameters

In [102]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


The token embedding and output layers are very large due to the number of rows for
the 50,257 in the tokenizer’s vocabulary. Let’s remove the output layer parameter
count from the total GPT-2 model count according to the weight tying:

In [103]:
total_params_gpt2 = (
    total_params - sum(p.numel()
    for p in model.out_head.parameters())
)
print(f"Number of trainable parameters "
      f"considering weight tying: {total_params_gpt2:,}"
)

Number of trainable parameters considering weight tying: 124,412,160


Lastly, let’s compute the memory requirements of the 163 million parameters in our
GPTModel object:

In [104]:
total_size_bytes = total_params * 4      
total_size_mb = total_size_bytes / (1024 * 1024)    
print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 621.83 MB


In [106]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    
    for _ in range(max_new_tokens):

        # Keep only the last `context_size` tokens as model input
        idx_cond = idx[:, -context_size:]

        # Disable gradient tracking (inference mode)
        with torch.no_grad():
            # Forward pass through the model
            # Output shape: (batch_size, n_tokens, vocab_size)
            logits = model(idx_cond)

        # Take logits from the last time step
        # Shape: (batch_size, vocab_size)
        logits = logits[:, -1, :]

        # Convert logits to probabilities
        probas = torch.softmax(logits, dim=-1)

        # Select the token with the highest probability (greedy decoding)
        # Shape: (batch_size, 1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)

        # Append the new token to the existing sequence
        idx = torch.cat((idx, idx_next), dim=1)

    return idx



generate_text_simple implements a basic generative loop for a GPT-style model.

The model generates text one token at a time and appends each new token to the context.

At each step:

logits are computed

softmax converts logits to probabilities

argmax selects the most likely next token (greedy decoding)

Softmax is monotonic, so applying argmax to logits or softmax outputs yields the same token.

Softmax is included for clarity and intuition.

Greedy decoding always selects the most likely token and lacks creativity.

More advanced sampling techniques (e.g., temperature, top-k) will be introduced later to improve diversity

In [107]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)   
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


<br/>

`model.eval()` switches the model to evaluation (inference) mode.

This disables training-only behaviors such as dropout.

Evaluation mode ensures stable and deterministic outputs.

The encoded input tensor is then passed to generate_text_simple to generate text

In [108]:
model.eval()
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
Output length: 10


Using the .decode method of the tokenizer, we can convert the IDs back into text

In [109]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am Featureiman Byeswickattribute argue


The generated text is gibberish because the model has not been trained.

So far, only the GPT architecture has been implemented.

The model’s weights are randomly initialized.

Without training, the model cannot produce coherent language.

Model training is a major topic and will be covered in the next chapter.